In [1]:
#read in 733 sample x 3.5+ million DHS presence/absence matrix
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle

In [ ]:
dhs = pd.read_table('dat_bin_FDR01_hg38.txt.gz', header=None).T
dhs.shape

In [3]:
data_coo = sp.sparse.coo_matrix(dhs.values)

In [4]:
print(data_coo)

  (0, 20)	1
  (0, 53)	1
  (0, 63)	1
  (0, 85)	1
  (0, 86)	1
  (0, 133)	1
  (0, 134)	1
  (0, 194)	1
  (0, 204)	1
  (0, 253)	1
  (0, 257)	1
  (0, 259)	1
  (0, 261)	1
  (0, 263)	1
  (0, 268)	1
  (0, 271)	1
  (0, 322)	1
  (0, 330)	1
  (0, 381)	1
  (0, 421)	1
  (0, 427)	1
  (0, 429)	1
  (0, 436)	1
  (0, 462)	1
  (0, 525)	1
  :	:
  (732, 3591784)	1
  (732, 3591786)	1
  (732, 3591787)	1
  (732, 3591790)	1
  (732, 3591791)	1
  (732, 3591793)	1
  (732, 3591794)	1
  (732, 3591797)	1
  (732, 3591799)	1
  (732, 3591803)	1
  (732, 3591805)	1
  (732, 3591806)	1
  (732, 3591810)	1
  (732, 3591813)	1
  (732, 3591814)	1
  (732, 3591815)	1
  (732, 3591816)	1
  (732, 3591817)	1
  (732, 3591819)	1
  (732, 3591820)	1
  (732, 3591824)	1
  (732, 3591825)	1
  (732, 3591832)	1
  (732, 3591834)	1
  (732, 3591837)	1


In [7]:
#splitting data into train and test
(train, test) = cross_validation.random_train_test_split(data_coo, test_percentage=0.2, random_state=None)
data = {'train': train, 'test': test}

In [8]:
print((data))

{'train': <733x3591898 sparse matrix of type '<class 'numpy.float32'>'
	with 61022029 stored elements in COOrdinate format>, 'test': <733x3591898 sparse matrix of type '<class 'numpy.int64'>'
	with 15255508 stored elements in COOrdinate format>}


In [3]:
#creating model
model30_d1 = LightFM(no_components=1, loss='warp')

In [4]:
#training model
model30_d1.fit(data['train'], epochs=1, num_threads=2)

In [12]:
pickle_out_model = open('DHS_model30.pickle', 'wb')
pickle.dump(model30, pickle_out_model)
pickle_out_model.close()

pickle_out_data = open('DHS_data30.pickle', 'wb')
pickle.dump(data, pickle_out_data)
pickle_out_data.close()

In [2]:
pickle_in = open('DHS_model30.pickle', 'rb')
model=pickle.load(pickle_in)

pickle_in = open('DHS_data30.pickle', 'rb')
data=pickle.load(pickle_in)

In [ ]:
# measure precision/recall @ k for k=1:50
test_precision = []
test_recall = []
train_precision = []
train_recall = []

for i in [1, 10, 100, 1000, 10000]:
    test_precision.append(precision_at_k(model30_d1, data['test'], k=i).mean())
    print('i')
#     test_recall.append(recall_at_k(model30, data['test'], k=i).mean())
#     train_precision.append(precision_at_k(model30, data['train'], k=i).mean())
#     train_recall.append(recall_at_k(model30, data['train'], k=i).mean())
#     print("k=" + i + " Finished")

In [12]:
item_biases, item_embeddings = model.get_item_representations(features=None)
user_biases, user_embeddings = model.get_user_representations(features=None)

In [13]:
print(item_embeddings.shape)
print(user_embeddings)

(3591898, 10)
[[ 2.594112   -2.7120328   1.3811816  ...  2.0611312  -2.729041
  -3.6168811 ]
 [ 1.4131172   3.722523   -2.1493971  ... -1.370422   -0.6829925
  -1.4827424 ]
 [ 2.2060719  -1.6619732   3.2524655  ...  0.2379665  -2.3723385
  -3.6720824 ]
 ...
 [ 3.6081746   1.5524385   1.6069187  ... -0.9677343  -0.09321057
  -1.9476378 ]
 [ 2.599359    0.3400904   3.399262   ...  1.7615211   2.4782457
  -2.4411473 ]
 [ 2.5906262  -0.39559594  3.5373619  ...  2.1963093   2.1646953
  -2.6249156 ]]


In [ ]:
auc = auc_score(model, data['test'])

In [6]:
n_biosamples, n_dhs = data['train'].shape

In [8]:
scores = model.predict(24, np.arange(n_dhs))
print(len(np.argsort(scores)))
#rank from most liked to least
top_items = [np.argsort(-scores)]

3591898


In [10]:
print(max(scores))

-59.630493


In [ ]:
model.predict_rank(data['test'])